# Spare-It POC: Synthetic Image Generator

In [1]:
import os
import json
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from pycocotools.coco import COCO
from matplotlib import image
from PIL import Image
from scipy import ndimage
import math
from skimage import measure
from imantics import Polygons, Mask

In [2]:
def crop(arr):
    slice_x, slice_y = ndimage.find_objects(arr>0)[0]
    return arr[slice_x, slice_y]
def cropc(arr):
    slice_x, slice_y, slice_z = ndimage.find_objects(arr>0)[0]
    return arr[slice_x, slice_y, slice_z]
def crop_coord(arr):
    slice_x, slice_y = ndimage.find_objects(arr>0)[0]
    return [slice_x, slice_y]
def apply_mask(image, mask):
    # Convert to numpy arrays
    mask = np.array(mask)
    # Convert grayscale image to RGB
    mask = np.stack((mask,)*3, axis=-1)
    # Multiply arrays
    resultant = image*mask
    return resultant

In [3]:
def max_file(path):
    files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
    max = 0
    for f in files:
        s = f.split('.')
        if(int(s[0]) > max):
            max = int(s[0])
    return max + 1

In [66]:
def random_offset(dist):
    center = dist/2
    var = dist/10
    rand = math.ceil(np.random.normal(center, var, 1)[0])
    if(rand < dist/4 or rand > 3*dist/4):
        return math.ceil(center)
    else:
        return rand

In [154]:
def copypaste(file1, file2, cid):
    dataDir='./cocojson'
    img_dir = './images'
    dataType='val'
    coco1=COCO('{}/'.format(dataDir,dataType) + file1)
    coco2=COCO('{}/'.format(dataDir,dataType) + file2)
    img1 = coco1.imgs[0]
    img2 = coco2.imgs[0]
    image1 = np.array(Image.open(os.path.join(img_dir, img1.get('file_name').split('images/')[-1])))
    image2 = np.array(Image.open(os.path.join(img_dir, img2.get('file_name').split('images/')[-1])))
    cat_ids = coco1.getCatIds()
    anns_ids1 = coco1.getAnnIds(imgIds=img1['id'], catIds=cat_ids, iscrowd=None)
    anns_ids2 = coco2.getAnnIds(imgIds=img2['id'], catIds=cat_ids, iscrowd=None)
    anns1 = coco1.loadAnns(anns_ids1)
    anns2 = coco2.loadAnns(anns_ids2)
    mask1 = np.zeros((img1['height'],img1['width']))
    mask2 = np.zeros((img2['height'],img2['width']))
    for i in range(len(anns1)):
        temp = coco1.annToMask(anns1[i])*anns1[i]['category_id']
        mask1 = np.where(temp != 0, temp, mask1)
    for i in range(len(anns2)):
        temp = coco2.annToMask(anns2[i])*anns2[i]['category_id']
        mask2 = np.where(temp != 0, temp, mask2)
    paste = np.where(mask1 == cid, 1, 0)
    mask_cropped = crop(paste)*cid
    image_cropped = cropc(apply_mask(image1, paste))
    x_offset=random_offset(img2['width'])
    y_offset=random_offset(img2['height'])
    y_inc = 0
    for i in range(y_offset,y_offset+image_cropped.shape[0]):
        x_inc = 0
        if(i < img2['height']):
            for j in range(x_offset,x_offset+image_cropped.shape[1]):
                if(mask_cropped[y_inc, x_inc] != 0 and j < img2['width']):
                    image2[i, j, :] = image_cropped[y_inc, x_inc, :]
                    mask2[i,j] = mask_cropped[y_inc, x_inc]
                x_inc +=1
        y_inc += 1
    filenum = max_file('./synthetic_images')
    newImg = Image.fromarray(image2)
    newImg.save('./synthetic_images/' + str(filenum) + '.jpeg')
    np.save('./masks/' + str(filenum), mask2)
    annotations = get_annotations(mask2)
    with open('./cocojson/' + file2, 'r+') as file:
        template = json.load(file)
    template['annotations'] = annotations
    with open('./synthetic_jsons/' + str(filenum) + '.json', 'w') as f:
        json.dump(template, f, indent=4)


#copypaste('Trash_fed220ac-f088-4708-9709-e23341f77d6a_74a6e7f7-4db5-41c6-9685-f3620b0d7a14_c4c093ab-ef17-45c0-9076-45b19d2a992c.json','Trash_feb7a7a0-7a9a-4ff0-b282-d68f14d9fe56_0f642e77-a98f-4840-94b0-fc9484670e14_77bbadc1-459c-4802-8c54-8502216b09f4.json', 1)

In [5]:
def mask_to_binary(mask):
    vals = np.unique(mask)
    z_index = np.where(vals == 0)
    vals = np.delete(vals, z_index)
    n = len(vals)
    binaries = []
    for i in range(n):
        binary = np.zeros(np.shape(mask))
        binary = np.where(mask == vals[i], 1, 0)
        binaries.append(binary)
    return binaries, vals
def get_annotations(mask):
    bins, cids = mask_to_binary(mask)
    n = len(cids)
    annotations = []
    inc = 0
    for i in range(n):
        polygons = Mask(bins[i]).polygons().segmentation
        for j in range(len(polygons)):
            ann = {}
            ann['id'] = inc
            inc += 1
            ann['image_id'] = 0
            ann['category_id'] = int(cids[i])
            polygon = polygons[j]
            seg = [polygon]
            ann['segmentation'] = seg
            annotations.append(ann)
    return annotations
            



In [25]:
files = [f for f in os.listdir('./cocojson/') if os.path.isfile(os.path.join('./cocojson/', f))]
with open('./cocojson/' + files[-1], 'r+') as file:
        template = json.load(file)
cats = template['categories']
categories = {}
for i in cats:
    categories[i['id']] = i['name']
files_by_id = {}
for i in categories:
    files_by_id[i] = set()


In [26]:
annList = template['annotations']
annList[0]['category_id']

126

In [30]:
i = 'BeverageCartons_276bca23-73f4-4441-9de3-27ae7028571a_7127adab-9712-405b-a0ff-79264b53b6a5_6150efc1-8844-4ff8-9909-40a760e48c48.json'
with open('./cocojson/' + i, 'r+') as file:
        temp = json.load(file)
cats = template['categories']
for i in cats:
    if(i['id'] == 106):
        print('wtf')

In [33]:
inc = 0
for i in files:
    with open('./cocojson/' + i, 'r+') as file:
        temp = json.load(file)
    annList = temp['annotations']
    for j in annList:
        cid = j['category_id']
        if(cid in categories):
            cats = temp['categories']
            
            cset = files_by_id[cid]
            cset.add(i)
            files_by_id[cid] = cset
    inc += 1
    if(inc % 1000 == 0):
        print(inc)

1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000


In [58]:
drops = []
for i in files_by_id:
    if(len(files_by_id[i]) == 0):
        drops.append(i)
for i in drops:
    files_by_id.pop(i)

In [105]:
probs = []
sums = []
ids = []
max = 0
for i in files_by_id:
    if(max < len(files_by_id[i])):
        max = len(files_by_id[i])
print(max)
for i in files_by_id:
    probs.append((max - len(files_by_id[i])))
    sums.append(len(files_by_id[i]))
    ids.append(i)
probs = probs/np.sum(probs)
probs

9701


array([0.00817466, 0.00538976, 0.01162768, 0.01160714, 0.01147062,
       0.01160714, 0.01169051, 0.0116893 , 0.01164581, 0.01166272,
       0.01163856, 0.01158419, 0.0108931 , 0.01022376, 0.01139571,
       0.01171105, 0.01167722, 0.005188  , 0.00946743, 0.01166393,
       0.01138846, 0.01051614, 0.01168809, 0.01171467, 0.00965107,
       0.01119756, 0.01165668, 0.01156244, 0.        , 0.01056568,
       0.01156365, 0.01119756, 0.01009569, 0.01156365, 0.00998091,
       0.01162406, 0.01169051, 0.01171951, 0.01170501, 0.01170138,
       0.01159869, 0.01171951, 0.00967403, 0.00961362, 0.01157935,
       0.01150565, 0.01097526, 0.01139208, 0.01163251, 0.01142712,
       0.01164581, 0.0116893 , 0.01169413, 0.01165547, 0.00819036,
       0.01094143, 0.01160593, 0.01146095, 0.01171467, 0.01122173,
       0.01155157, 0.01130872, 0.00917867, 0.01153465, 0.00805625,
       0.01151049, 0.01109366, 0.01068408, 0.00755244, 0.01067079,
       0.01119273, 0.01112991, 0.01166634, 0.01156727, 0.01117

In [161]:
def random_sampling(k, probs):
    for i in range(k):
        rid1 = random.choices(ids, probs)[0]
        rfile1 = random.choice(tuple(files_by_id[rid1]))
        rfile2 = random.choice(tuple(files_by_id[random.choice(ids)]))
        try:
            copypaste(rfile1, rfile2, rid1)
        except:
            print('copypaste error')
random_sampling(100, probs)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
load